# Confidence-gated memory guidance for invoice OCR

This notebook builds the vendor-rooted **memory graph** in Neo4j from ground-truth invoice
extractions. It then compares two ways of letting that memory guide a Gemini (Vertex AI) OCR model on held-out PDFs:

* **Pass 1, ungated and guided:** Pick the layout whose structure vector is closest by cosine similarity, and hand all
  of its rules to Gemini.
* **Pass 2, gated.** Compute a **posterior probability over remembered layouts** from what an unguided extraction
  of the page shows: its vendor name, which fields it has, and where their boxes sit. Then compute, for every rule,
  the probability that it is right *for this page*. Only rules above a threshold reach Gemini, together with their
  probability.

Both passes are built from the **same unguided extraction** (call 0), so any difference between them comes from the
gate. Call 0 is evidence only, so it is saved but not scored.

**Confidence gating.** A rule is shown only when its probability of applying to this document passes a threshold.
Otherwise a weaker fallback is used: a label hint instead of a position, or nothing at all.

### Configuration
Every setting lives in this cell.

In [ ]:
from pathlib import Path

# --- Vertex AI ---------------------------------------------------------------------------------------------------
GCP_PROJECT = "rahhh"
GCP_LOCATION = "global"
OCR_MODEL = "gemini-2.5-flash"

# --- Neo4j (its own container, so resetting this graph leaves graph-grounding-v2's graph alone) ------------------
NEO4J_URI = "bolt://localhost:7688"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "gating-local"
RESET_GRAPH = True

# --- Data --------------------------------------------------------------------------------------------------------
DATA_DIR = Path.cwd().parent / "data"   # folder holding input/ and output/; here, next to the project folder
INPUT_DIR = DATA_DIR / "input"          # PDFs
OUTPUT_DIR = DATA_DIR / "output"        # ground-truth JSON
RESULTS_DIR = Path("results")           # unguided call, pass outputs, guides

# --- Split -------------------------------------------------------------------------------------------------------
TRAIN_FRACTION = 2 / 3                  # share of each vendor's documents used to build the graph
SPLIT_SEED = 7

# --- Consolidation (as in v2) ------------------------------------------------------------------------------------
STRUCTURE_SIMILARITY = 0.85             # cosine of fill vectors at or above which two pages share a layout variant
BBOX_TOLERANCE = 0.03                   # box centres within this many page-widths/heights count as "the same spot"
MIN_SUPPORT_FRACTION = 0.6              # share of located observations that must agree on the spot to make a rule
MIN_SUPPORT_DOCS = 2                    # ...and at least this many documents must agree
STRONG_SUPPORT_DOCS = 3                 # a consolidated rule backed by this many documents is "strong"

# --- Position bins and box ranges --------------------------------------------------------------------------------
POSITION_GRID = 4                       # the page is cut into a 4 x 4 grid for the position part of the vector
POSITION_WEIGHT = 1.0                   # weight of the position part against the fill part in cosine similarity
BOX_RANGE_MIN = 0.06                    # smallest half-width of a rule's range; v2's unguided boxes missed by ~0.057
BOX_RANGE_MAX = 0.15                    # largest half-width, so an entity that moves around gets a low p_in_range
BOX_RANGE_SPREAD_K = 2.0                # otherwise the half-width is this many bbox_spreads

# --- Posterior over layouts --------------------------------------------------------------------------------------
FIELD_SMOOTHING_M = 2.0                 # pseudo-pages of corpus behaviour mixed into every layout's fill rates
FIELD_MISS = 0.05                       # P(extraction leaves a printed field empty)
FIELD_FALSE_FILL = 0.02                 # P(extraction fills a field that is not printed); v2 measured 0.023
NAME_ERROR = 0.10                       # P(the extracted vendor name maps to the wrong vendor key)
NAME_VARIANT = 0.5                      # share of those errors that are a variant spelling of the true key
FIELD_TEMPERATURE = 1.0                 # > 1 flattens the field term (fields are not independent)
RULE_PRIOR_STRENGTH = 2.0               # pseudo-trials of the empirical-Bayes prior on a rule's in-range rate

# --- Gate --------------------------------------------------------------------------------------------------------
GATE_BOX = 0.7                          # confidence at or above which a rule's typical box is shown
GATE_LABEL = 0.4                        # ...at or above which only its label is shown; below it, it is withheld

# --- Guide and test run ------------------------------------------------------------------------------------------
GUIDE_NEIGHBOURS = 2                    # structurally similar layouts offered by the ungated guide when the vendor is unknown
LIMIT_TEST_DOCS = None                  # e.g. 3 for a smoke test; None runs the whole held-out third
TOP_DIFF_DOCS = 5                       # documents printed in the pass-1 vs pass-2 comparison

#### Neo4j setup
This project uses its own container, on ports 7475/7688:
```
docker run --name graph_gating -d \
  -p 7475:7474 -p 7688:7687 \
  -v $HOME/Neo/graph_gating/neo4j/data:/data:Z \
  -v $HOME/Neo/graph_gating/neo4j/logs:/logs:Z \
  -e NEO4J_AUTH=neo4j/gating-local \
  neo4j:5.26-community
```

## Load the ground truth and split by vendor
`helpers.flatten_document` turns each ground-truth JSON into a flat record. Per page, it holds every key-value
entity with its value, its box as **page fractions** (`[x, y, w, h]`, 0–1, top-left origin) and the label printed
next to it, plus the table rows. `helpers.pdf_page_facts` reads page sizes and words from the PDFs. The JSON lacks
them; see the future data requirements in `helpers.py`.

As in v2, each vendor's training share is proportional to its set size (largest-remainder rounding, seeded draw
within a vendor). A vendor with a single document goes entirely to one side. If it lands in test, it is a
**vendor the graph has never seen**.

In [ ]:
import json
import math
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import helpers as H

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

doc_ids = sorted(p.stem for p in OUTPUT_DIR.glob("*.json") if (INPUT_DIR / f"{p.stem}.pdf").exists())
assert doc_ids, f"no ground-truth JSON with a matching PDF under {DATA_DIR.resolve()}: set DATA_DIR in the config cell"
ground_truth = {d: json.loads((OUTPUT_DIR / f"{d}.json").read_text()) for d in doc_ids}   # raw JSON
page_facts = {d: H.pdf_page_facts(INPUT_DIR / f"{d}.pdf") for d in doc_ids}               # sizes + words
records = {d: H.flatten_document(d, ground_truth[d], page_facts[d]) for d in doc_ids}     # flat GT records

print(f"{len(records)} documents, {sum(r['page_count'] for r in records.values())} pages, "
      f"{sum(r['is_scanned'] for r in records.values())} scans")

In [ ]:
def vendor_table(records: dict) -> pd.DataFrame:
    """One row per vendor root: its set size, pages, scans and the spelling variants merged under it."""
    rows = defaultdict(lambda: {"names": set(), "n_docs": 0, "n_pages": 0, "n_scanned": 0})
    for rec in records.values():
        row = rows[rec["vendor_key"]]
        row["names"].add(rec["vendor_name"] or "(none)")
        row["n_docs"] += 1
        row["n_pages"] += rec["page_count"]
        row["n_scanned"] += rec["is_scanned"]
    df = pd.DataFrame([{"vendor_key": k, "name_variants": " | ".join(sorted(v["names"])), **{c: v[c] for c in
                        ("n_docs", "n_pages", "n_scanned")}} for k, v in rows.items()])
    return df.sort_values(["n_docs", "vendor_key"], ascending=[False, True]).reset_index(drop=True)


def proportional_split(records: dict, fraction: float, seed: int) -> tuple[dict[str, int], set[str], set[str]]:
    """Training count per vendor (largest-remainder rounding to an exact corpus total), then a seeded random draw
    of which documents within each vendor go to training. Returns (n_train per vendor, train ids, test ids)."""
    rng = random.Random(seed)
    by_vendor = defaultdict(list)
    for doc_id in sorted(records):
        by_vendor[records[doc_id]["vendor_key"]].append(doc_id)
    ideal = {v: fraction * len(ids) for v, ids in by_vendor.items()}
    n_train = {v: int(x) for v, x in ideal.items()}
    leftover = round(fraction * len(records)) - sum(n_train.values())
    tie_break = {v: rng.random() for v in by_vendor}
    for v in sorted(ideal, key=lambda v: (ideal[v] - n_train[v], tie_break[v]), reverse=True)[:leftover]:
        n_train[v] += 1
    train, test = set(), set()
    for v, ids in sorted(by_vendor.items()):
        chosen = set(rng.sample(ids, n_train[v]))
        train |= chosen
        test |= set(ids) - chosen
    return n_train, train, test


def size_bucket(n_train: int) -> str:
    return "0 (unseen)" if n_train == 0 else "1" if n_train == 1 else "2-3" if n_train <= 3 else \
           "4-9" if n_train <= 9 else "10+"


BUCKET_ORDER = ["0 (unseen)", "1", "2-3", "4-9", "10+"]
vendors_df = vendor_table(records)
_, train_ids, test_ids = proportional_split(records, TRAIN_FRACTION, SPLIT_SEED)
train_count = Counter(records[d]["vendor_key"] for d in train_ids)
test_count = Counter(records[d]["vendor_key"] for d in test_ids)
vendors_df["n_train"] = vendors_df["vendor_key"].map(lambda v: train_count[v])
vendors_df["n_test"] = vendors_df["vendor_key"].map(lambda v: test_count[v])
vendors_df["train_set_size_bucket"] = vendors_df["n_train"].map(size_bucket)

# Sanity checks: disjoint, exact totals, each vendor within one document of its proportional share
assert not train_ids & test_ids and len(train_ids) + len(test_ids) == len(records)
assert len(train_ids) == round(TRAIN_FRACTION * len(records))
assert (abs(vendors_df.n_train - TRAIN_FRACTION * vendors_df.n_docs) < 1).all()

print(f"{len(vendors_df)} vendor roots | train {len(train_ids)} | test {len(test_ids)} "
      f"({sum(train_count[records[d]['vendor_key']] > 0 for d in test_ids)} test docs from seen vendors)")
display(vendors_df.head(15))
vendors_df.groupby("train_set_size_bucket").agg(vendors=("vendor_key", "count"), train_docs=("n_train", "sum"),
                                               test_docs=("n_test", "sum")).reindex(BUCKET_ORDER)

## The memory graph
The graph is v2's single layer of memory paths, with two additions marked **new**.

```
(:DocumentType {name:"Invoice"})
      ▲ OF_TYPE
(:Vendor {key})
      │ HAS_LAYOUT
(:PageLayout {page_number, variant, fill_vector, structure_vector})
      ├─HAS_SECTION {json_position}─►(:SectionLayout {name})
      │        └─HAS_ENTITY {json_position}─►(:EntityLayout {key, obs_*, rule…, bbox_range, p_in_range})
      │                 ├─NORMALIZED_BY─►(:NormalizationRule {name})
      │                 └─CONSTRAINED_BY─►(:Invariant {name})
      └─HAS_TABLE─►(:TableLayout {type})
               └─HAS_COLUMN {json_position}─►(:ColumnLayout {header})
```

| Node | Property | Meaning |
|---|---|---|
| `Vendor` | `key`, `names`, `doc_count`, `train_doc_ids` | normalized vendor name and its training set. `ALL_VENDORS` is the corpus-wide tree that every training page is also merged into |
| `PageLayout` | `id`, `page_number`, `variant`, `doc_count`, `doc_ids` | one layout variant of one page number of one vendor |
| | `fill_vector` | mean fill vector: the share of pages where each entity is filled and each table is present |
| | `structure_vector` **(new)** | `fill_vector` followed by the mean **position bins** of every entity (below) |
| `EntityLayout` | `obs_*` | one observation per merged page: filled flag, value box, printed label |
| | rule properties | written by consolidation: `fill_rate`, `support`, `bbox_center`, `bbox_median`, `bbox_spread`, `top_labels`, `strength`, `order` |
| | `bbox_range`, `n_in_range`, `p_in_range` **(new)** | the rule's approximate range and how reliably values fall inside it (consolidation) |
| `TableLayout`, `ColumnLayout` | `obs_*`, `presence_rate`, `region_bbox`, `fill_rate`, `x_band` | as in v2 |

**Position bins.** `helpers.soft_bins` cuts the page into a `POSITION_GRID × POSITION_GRID` grid and spreads a box
centre over the four nearest cell centres with bilinear weights. A centre near a cell edge is shared with the
neighbouring cell. A small shift caused by a sloppy extracted box therefore changes the vector only a little,
instead of flipping it from one cell to another. `helpers.structure_vector(page)` is the fill vector followed by
`POSITION_WEIGHT ×` the soft bins of each entity. Cosine similarity of structure vectors compares *which* fields a
page has and *roughly where* they are.

**Variants** are still formed from fill vectors alone, exactly as in v2: pages of one vendor and page number share a
variant when their cosine similarity is at least `STRUCTURE_SIMILARITY`. Positions refine the comparison at
guide time; they don't split the training data further.

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD), notifications_min_severity="OFF")
driver.verify_connectivity()


def cypher(query: str, **params) -> list[dict]:
    records_, _, _ = driver.execute_query(query, **params)
    return [r.data() for r in records_]


def cypher_df(query: str, **params) -> pd.DataFrame:
    return pd.DataFrame(cypher(query, **params))


ALL_VENDORS = "ALL_VENDORS"

if RESET_GRAPH:
    cypher("MATCH (n) DETACH DELETE n")
for label, prop in [("Vendor", "key"), ("PageLayout", "id"), ("SectionLayout", "id"), ("EntityLayout", "id"),
                    ("TableLayout", "id"), ("ColumnLayout", "id"), ("NormalizationRule", "name"),
                    ("Invariant", "name"), ("DocumentType", "name")]:
    cypher(f"CREATE CONSTRAINT {label.lower()}_{prop} IF NOT EXISTS FOR (n:{label}) REQUIRE n.{prop} IS UNIQUE")

# Special nodes from the schema
cypher("MERGE (:DocumentType {name: $name})", name=H.DOC_TYPE)
cypher("UNWIND $rules AS r MERGE (n:NormalizationRule {name: r.name}) SET n.rule = r.rule",
       rules=[{"name": k, "rule": v} for k, v in H.NORMALIZATION_RULES.items()])
cypher("MERGE (i:Invariant {name: 'arithmetic_check'}) SET i.expression = $expr", expr=H.ARITHMETIC_INVARIANT)
print("Neo4j:", cypher("CALL dbms.components() YIELD versions RETURN versions[0] AS v")[0]["v"])

In [ ]:
def assign_variants(records: dict, doc_ids: set[str], threshold: float) -> dict[tuple[str, int], int]:
    """(doc_id, page number) -> variant index within (vendor, page number).

    Greedy, deterministic grouping: pages are visited in doc_id order and join the first variant whose running
    mean fill vector has cosine >= threshold with theirs, otherwise they open a new variant."""
    groups = defaultdict(list)                     # (vendor, page) -> [(centroid, count)]
    variant = {}
    for doc_id in sorted(doc_ids):
        rec = records[doc_id]
        for page in rec["pages"]:
            vec = np.array(H.fill_vector(page))
            slots = groups[(rec["vendor_key"], page["number"])]
            best = max(range(len(slots)), key=lambda i: H.cosine(slots[i][0] / slots[i][1], vec), default=None)
            if best is not None and H.cosine(slots[best][0] / slots[best][1], vec) >= threshold:
                slots[best] = (slots[best][0] + vec, slots[best][1] + 1)
            else:
                slots.append((vec.copy(), 1))
                best = len(slots) - 1
            variant[(doc_id, page["number"])] = best
    return variant


variants = assign_variants(records, train_ids, STRUCTURE_SIMILARITY)
print(f"{len(set((records[d]['vendor_key'], n, v) for (d, n), v in variants.items()))} page layouts from "
      f"{len(variants)} training pages")

### Ingestion: each training document walks its path
`ingest_page` `MERGE`s the path Vendor → PageLayout → SectionLayout → EntityLayout (and → TableLayout →
ColumnLayout), then **appends** the page's observation to the arrays on each node. A guard (`doc_id IN
obs_doc_ids`) makes re-runs idempotent. Every page goes into its own vendor's tree and into `ALL_VENDORS`. Besides
`fill_sum`, the PageLayout now accumulates `structure_sum`, and `structure_vector` is its mean.

In [ ]:
INGEST_PAGE = """
MERGE (v:Vendor {key: $vendor})
  ON CREATE SET v.names = [], v.train_doc_ids = []
WITH v, CASE WHEN $doc_id IN v.train_doc_ids THEN v.train_doc_ids ELSE v.train_doc_ids + $doc_id END AS doc_ids
SET v.names = CASE WHEN $vendor_name IN v.names OR $vendor_name = '' THEN v.names ELSE v.names + $vendor_name END,
    v.train_doc_ids = doc_ids, v.doc_count = size(doc_ids)
WITH v
MATCH (t:DocumentType {name: $doc_type})
MERGE (v)-[:OF_TYPE]->(t)
MERGE (v)-[:HAS_LAYOUT]->(p:PageLayout {id: $layout_id})
  ON CREATE SET p.page_number = $page_number, p.variant = $variant, p.doc_ids = [], p.fill_sum = [],
                p.structure_sum = []
WITH p, NOT ($doc_id IN p.doc_ids) AS is_new
WITH p, is_new, CASE WHEN is_new THEN p.doc_ids + $doc_id ELSE p.doc_ids END AS doc_ids,
     CASE WHEN NOT is_new THEN p.fill_sum
          WHEN size(p.fill_sum) = 0 THEN $fill_vector
          ELSE [i IN range(0, size($fill_vector) - 1) | p.fill_sum[i] + $fill_vector[i]] END AS fill_sum,
     CASE WHEN NOT is_new THEN p.structure_sum
          WHEN size(p.structure_sum) = 0 THEN $structure_vector
          ELSE [i IN range(0, size($structure_vector) - 1) | p.structure_sum[i] + $structure_vector[i]] END
       AS structure_sum
SET p.doc_ids = doc_ids, p.doc_count = size(doc_ids), p.fill_sum = fill_sum,
    p.fill_vector = [x IN fill_sum | x / size(doc_ids)],
    p.structure_sum = structure_sum, p.structure_vector = [x IN structure_sum | x / size(doc_ids)]
WITH p, is_new
UNWIND $fields AS f
MERGE (p)-[:HAS_SECTION {json_position: f.section_position}]->(s:SectionLayout {id: $layout_id + '|' + f.section})
  ON CREATE SET s.name = f.section
MERGE (s)-[:HAS_ENTITY {json_position: f.key_position}]->(e:EntityLayout {id: $layout_id + '|' + f.key})
  ON CREATE SET e.key = f.key, e.obs_doc_ids = [], e.obs_filled = [], e.obs_x = [], e.obs_y = [], e.obs_w = [],
                e.obs_h = [], e.obs_labels = []
WITH e, f, is_new WHERE is_new
SET e.obs_doc_ids = e.obs_doc_ids + $doc_id, e.obs_filled = e.obs_filled + f.filled,
    e.obs_x = e.obs_x + f.x, e.obs_y = e.obs_y + f.y, e.obs_w = e.obs_w + f.w, e.obs_h = e.obs_h + f.h,
    e.obs_labels = e.obs_labels + f.label
"""

INGEST_TABLES = """
MATCH (p:PageLayout {id: $layout_id})
UNWIND $tables AS t
MERGE (p)-[:HAS_TABLE]->(tl:TableLayout {id: $layout_id + '|' + t.type})
  ON CREATE SET tl.type = t.type, tl.obs_doc_ids = [], tl.obs_rows = [], tl.obs_x = [], tl.obs_y = [],
                tl.obs_w = [], tl.obs_h = []
WITH tl, t, NOT ($doc_id IN tl.obs_doc_ids) AS is_new
SET tl.obs_doc_ids = CASE WHEN is_new THEN tl.obs_doc_ids + $doc_id ELSE tl.obs_doc_ids END,
    tl.obs_rows = CASE WHEN is_new THEN tl.obs_rows + t.rows ELSE tl.obs_rows END,
    tl.obs_x = CASE WHEN is_new THEN tl.obs_x + t.region[0] ELSE tl.obs_x END,
    tl.obs_y = CASE WHEN is_new THEN tl.obs_y + t.region[1] ELSE tl.obs_y END,
    tl.obs_w = CASE WHEN is_new THEN tl.obs_w + t.region[2] ELSE tl.obs_w END,
    tl.obs_h = CASE WHEN is_new THEN tl.obs_h + t.region[3] ELSE tl.obs_h END
WITH tl, t, is_new
UNWIND t.columns AS c
MERGE (tl)-[:HAS_COLUMN {json_position: c.position}]->(cl:ColumnLayout {id: tl.id + '|' + c.header})
  ON CREATE SET cl.header = c.header, cl.obs_filled = [], cl.obs_x = []
WITH cl, c, is_new WHERE is_new
SET cl.obs_filled = cl.obs_filled + c.filled, cl.obs_x = cl.obs_x + c.x
"""


def table_payload(page: dict) -> list[dict]:
    """Per table type on the page: row count, the region its cell boxes cover, and per column the filled flags
    and x-centres of its cells (-1 when a cell has no box)."""
    payload = []
    for table in page["tables"]:
        columns = H.TABLE_TYPES[table["type"]]
        boxes = [c["bbox"] for row in table["rows"] for c in row.values() if c["bbox"]]
        if boxes:
            x0, y0 = min(b[0] for b in boxes), min(b[1] for b in boxes)
            x1, y1 = max(b[0] + b[2] for b in boxes), max(b[1] + b[3] for b in boxes)
            region = [x0, y0, x1 - x0, y1 - y0]
        else:
            region = [-1.0, -1.0, -1.0, -1.0]
        payload.append({"type": table["type"], "rows": len(table["rows"]), "region": region, "columns": [
            {"header": col, "position": i + 1,
             "filled": [bool(row.get(col, {}).get("value")) for row in table["rows"]],
             "x": [H.bbox_center(row[col]["bbox"])[0] if row.get(col, {}).get("bbox") else -1.0
                   for row in table["rows"]]}
            for i, col in enumerate(columns)]})
    return payload


def ingest_page(rec: dict, page: dict, vendor: str, variant: int) -> None:
    layout_id = f"{vendor}|p{page['number']}|v{variant}"
    section_position = {name: i + 1 for i, name in enumerate(H.SECTIONS)}
    fields = []
    for key in H.KV_KEYS:
        f = page["fields"].get(key, {"value": "", "bbox": None, "label": ""})
        x, y, w, h = f["bbox"] or [-1.0, -1.0, -1.0, -1.0]
        section = H.SECTION_OF[key]
        fields.append({"key": key, "section": section, "section_position": section_position[section],
                       "key_position": H.SECTIONS[section].index(key) + 1, "filled": bool(f["value"]),
                       "x": x, "y": y, "w": w, "h": h, "label": f["label"]})
    cypher(INGEST_PAGE, vendor=vendor, vendor_name=rec["vendor_name"] if vendor != ALL_VENDORS else "",
           doc_id=rec["doc_id"], doc_type=H.DOC_TYPE, layout_id=layout_id, page_number=page["number"],
           variant=variant, fill_vector=H.fill_vector(page),
           structure_vector=H.structure_vector(page, POSITION_GRID, POSITION_WEIGHT), fields=fields)
    # "General" has no entities but is still part of every page's path
    cypher("""MATCH (p:PageLayout {id: $layout_id})
              MERGE (p)-[:HAS_SECTION {json_position: $pos}]->(s:SectionLayout {id: $layout_id + '|General'})
              SET s.name = 'General'""", layout_id=layout_id, pos=section_position["General"])
    cypher(INGEST_TABLES, layout_id=layout_id, doc_id=rec["doc_id"], tables=table_payload(page))


for doc_id in sorted(train_ids):
    rec = records[doc_id]
    for page in rec["pages"]:
        ingest_page(rec, page, rec["vendor_key"], variants[(doc_id, page["number"])])
        ingest_page(rec, page, ALL_VENDORS, 0)

# Special-node links: each entity to its normalization rule, invariant members to the invariant
cypher("""MATCH (e:EntityLayout) MATCH (r:NormalizationRule {name: $types[e.key]})
          MERGE (e)-[:NORMALIZED_BY]->(r)""", types=H.VALUE_TYPES)
cypher("""MATCH (n) WHERE (n:EntityLayout AND n.key IN $members) OR (n:ColumnLayout AND n.header IN $members)
          MATCH (i:Invariant {name: 'arithmetic_check'}) MERGE (n)-[:CONSTRAINED_BY]->(i)""",
       members=sorted(H.INVARIANT_MEMBERS))

# Checks: every training document sits in exactly one vendor tree; one observation per merged page
in_trees = cypher("MATCH (v:Vendor) WHERE v.key <> $all UNWIND v.train_doc_ids AS d RETURN d", all=ALL_VENDORS)
assert sorted(r["d"] for r in in_trees) == sorted(train_ids), "a training document is missing or duplicated"
bad = cypher("""MATCH (p:PageLayout)-[:HAS_SECTION]->()-[:HAS_ENTITY]->(e)
                WHERE size(e.obs_doc_ids) <> p.doc_count OR size(e.obs_x) <> p.doc_count RETURN e.id AS id""")
assert not bad, bad[:3]
cypher_df("MATCH (n) RETURN labels(n)[0] AS node, count(*) AS count ORDER BY node")

## Consolidation: rules, ranges and how much to trust them
For one entity in one layout:

1. Every usable box centre is a candidate. The **peak** is the centre with the most other centres within
   `BBOX_TOLERANCE` on both axes. Its **support** is that count, and `support_fraction` is support divided by
   the number of usable boxes.
2. The entity is **consolidated** when `support_fraction ≥ MIN_SUPPORT_FRACTION` and `support ≥ MIN_SUPPORT_DOCS`.
   It is **strong** when, in addition, `support ≥ STRONG_SUPPORT_DOCS`, and **weak** otherwise. A one-document
   layout gets **single** rules.
3. `bbox_median` is the coordinate-wise median of the supporting boxes, `bbox_spread` the median distance of all
   centres from the peak, and `top_labels` the most frequent printed labels.

**New: the approximate range.** Gemini's boxes are noisy: unguided centres missed the ground-truth centre
by 0.057 page-widths on average, about twice `BBOX_TOLERANCE`. A point-like rule can't be matched against such
boxes, so each rule also gets a square **range** around its peak:

$$R_{L,k} = [c_x - m,\ c_x + m] \times [c_y - m,\ c_y + m], \qquad
m = \mathrm{clip}\big(\texttt{BOX\_RANGE\_SPREAD\_K} \cdot \texttt{bbox\_spread},\ \texttt{BOX\_RANGE\_MIN},\ \texttt{BOX\_RANGE\_MAX}\big)$$

`n_in_range` counts the training centres that fall inside $R_{L,k}$.

**New: `p_in_range`, the rule's reliability as a probability.** Let $q_{L,k}$ be the unknown probability that,
on a new page of layout $L$, the value of entity $k$ (when printed) has its centre inside $R_{L,k}$. Each
training page with a usable box is one Bernoulli trial: in or out. With a $\mathrm{Beta}(\alpha, \beta)$ prior,
the posterior after $s$ hits out of $n$ trials is $\mathrm{Beta}(\alpha + s,\ \beta + n - s)$, and the
probability that the *next* page is a hit is its mean, $(s + \alpha)/(n + \alpha + \beta)$.

**The peak is not evidence.** The range is centred on the peak box, so that box is inside it by construction.
Counting it as a hit would claim evidence that was never at risk. A one-document rule would "succeed" 1 of 1
times without having predicted anything. So the trials are the $n - 1$ boxes other than the peak, with $s - 1$
hits:

$$P(c_k \in R_{L,k} \mid L,\ \text{training pages}) = \frac{(s - 1) + \alpha}{(n - 1) + \alpha + \beta}$$

**An empirical-Bayes prior.** A Laplace prior $\mathrm{Beta}(1, 1)$ says that a rule we know nothing about is
right half the time. On the first run that was far too pessimistic: one-document rules were rated 0.67 and hit
0.87. The prior's mean $p_0$ is instead estimated from the training data. For every vendor layout with at least two
located boxes, take every ordered pair of its boxes $(i, j)$, and ask whether centre $j$ lies within
`BOX_RANGE_MIN` of centre $i$ on both axes. That is exactly the question "would a rule learned from document
$i$ alone have caught document $j$?", the out-of-sample hit rate of a one-document rule. The pair hit rates are
averaged per layout, then over layouts, so the one large vendor doesn't dominate. The prior is then
$\alpha = s_0\,p_0,\ \beta = s_0(1 - p_0)$, with $s_0$ = `RULE_PRIOR_STRENGTH` pseudo-trials:

| Training boxes | Hits besides the peak | `p_in_range` |
|---|---|---|
| 1 | — | $p_0$ |
| 3 | 2 of 2 | $(2 + s_0 p_0)/(2 + s_0)$ |
| 14 | 13 of 13 | $(13 + s_0 p_0)/(13 + s_0)$ |

v2's hard labels (single / weak / strong) become one number that grows with evidence. This number is
$P(\text{rule right} \mid \text{the page really is } L)$. It doesn't yet say whether the page *is* $L$; that is
the job of the posterior further below.

In [ ]:
CONSOLIDATE_ENTITIES = """
MATCH (p:PageLayout)-[:HAS_SECTION]->()-[:HAS_ENTITY]->(e:EntityLayout)
WITH p, e, [i IN range(0, size(e.obs_x) - 1) WHERE e.obs_x[i] >= 0 |
            {x: e.obs_x[i], y: e.obs_y[i], w: e.obs_w[i], h: e.obs_h[i],
             cx: e.obs_x[i] + e.obs_w[i] / 2, cy: e.obs_y[i] + e.obs_h[i] / 2}] AS boxes
// 1. density peak: the box centre with the most neighbours within the tolerance
WITH p, e, boxes, [b IN boxes | size([o IN boxes WHERE abs(o.cx - b.cx) <= $tol AND abs(o.cy - b.cy) <= $tol])]
     AS density
WITH p, e, boxes, density, reduce(best = -1, i IN range(0, size(boxes) - 1) |
     CASE WHEN best = -1 OR density[i] > density[best] THEN i ELSE best END) AS peak
WITH p, e, boxes, CASE WHEN peak >= 0 THEN boxes[peak] END AS centre, CASE WHEN peak >= 0 THEN density[peak]
     ELSE 0 END AS support
WITH p, e, boxes, centre, support,
     [b IN boxes WHERE centre IS NOT NULL AND abs(b.cx - centre.cx) <= $tol AND abs(b.cy - centre.cy) <= $tol]
     AS members
// 2. medians of the supporting boxes, and the spread of all boxes around the peak
CALL (members, boxes, centre) {
  UNWIND (CASE WHEN size(members) = 0 THEN [null] ELSE members END) AS m
  WITH boxes, centre, percentileCont(m.x, 0.5) AS mx, percentileCont(m.y, 0.5) AS my,
       percentileCont(m.w, 0.5) AS mw, percentileCont(m.h, 0.5) AS mh
  UNWIND (CASE WHEN size(boxes) = 0 THEN [null] ELSE boxes END) AS b
  RETURN mx, my, mw, mh, percentileCont(sqrt((b.cx - centre.cx)^2 + (b.cy - centre.cy)^2), 0.5) AS spread
}
WITH p, e, boxes, centre, support, mx, my, mw, mh, spread,
     size(e.obs_filled) AS n_obs, size([f IN e.obs_filled WHERE f]) AS n_filled
// 3. labels
CALL (e) {
  UNWIND (CASE WHEN size([l IN e.obs_labels WHERE l <> '']) = 0 THEN [null] ELSE
          [l IN e.obs_labels WHERE l <> ''] END) AS label
  WITH label, count(label) AS n ORDER BY n DESC
  RETURN [l IN collect(label) WHERE l IS NOT NULL][0..3] AS top_labels
}
WITH e, p, n_obs, n_filled, size(boxes) AS n_boxes, support, centre, mx, my, mw, mh, spread, top_labels,
     CASE WHEN size(boxes) = 0 THEN 0.0 ELSE toFloat(support) / size(boxes) END AS support_fraction
WITH e, p, n_obs, n_filled, n_boxes, support, centre, mx, my, mw, mh, spread, top_labels, support_fraction,
     support_fraction >= $min_fraction AND support >= $min_docs AS consolidated
SET e.n_obs = n_obs, e.n_filled = n_filled, e.n_boxes = n_boxes,
    e.fill_rate = round(toFloat(n_filled) / n_obs, 3),
    e.support = support, e.support_fraction = round(support_fraction, 3),
    e.bbox_center = CASE WHEN centre IS NULL THEN null ELSE [round(centre.cx, 4), round(centre.cy, 4)] END,
    e.bbox_median = CASE WHEN centre IS NULL THEN null ELSE [round(mx, 4), round(my, 4), round(mw, 4), round(mh, 4)] END,
    e.bbox_spread = CASE WHEN spread IS NULL THEN null ELSE round(spread, 4) END,
    e.top_labels = top_labels,
    e.consolidated = consolidated,
    e.strength = CASE WHEN consolidated AND support >= $strong_docs THEN 'strong'
                      WHEN consolidated THEN 'weak'
                      WHEN p.doc_count = 1 AND centre IS NOT NULL THEN 'single'
                      ELSE 'none' END
"""

# 4. the approximate range around the peak, how many training centres it holds, and the Beta-Binomial
#    posterior predictive probability that the next centre falls inside it (the peak itself is not counted)
CONSOLIDATE_RANGES = """
MATCH (e:EntityLayout)
WITH e, CASE WHEN e.bbox_center IS NULL THEN null
             WHEN $k * e.bbox_spread < $min_margin THEN $min_margin
             WHEN $k * e.bbox_spread > $max_margin THEN $max_margin
             ELSE $k * e.bbox_spread END AS m
WITH e, CASE WHEN m IS NULL THEN null ELSE
     [CASE WHEN e.bbox_center[0] - m < 0 THEN 0.0 ELSE e.bbox_center[0] - m END,
      CASE WHEN e.bbox_center[1] - m < 0 THEN 0.0 ELSE e.bbox_center[1] - m END,
      CASE WHEN e.bbox_center[0] + m > 1 THEN 1.0 ELSE e.bbox_center[0] + m END,
      CASE WHEN e.bbox_center[1] + m > 1 THEN 1.0 ELSE e.bbox_center[1] + m END] END AS r
WITH e, r, CASE WHEN r IS NULL THEN 0 ELSE size([i IN range(0, size(e.obs_x) - 1) WHERE e.obs_x[i] >= 0
          AND r[0] <= e.obs_x[i] + e.obs_w[i] / 2 AND e.obs_x[i] + e.obs_w[i] / 2 <= r[2]
          AND r[1] <= e.obs_y[i] + e.obs_h[i] / 2 AND e.obs_y[i] + e.obs_h[i] / 2 <= r[3]]) END AS n_in
SET e.bbox_range = CASE WHEN r IS NULL THEN null ELSE [x IN r | round(x, 4)] END, e.n_in_range = n_in,
    e.p_in_range = CASE WHEN r IS NULL THEN null                 // the peak defines the range: not a trial
                        ELSE (n_in - 1 + $alpha) / (e.n_boxes - 1 + $alpha + $beta) END
"""

ORDER_ENTITIES = """
MATCH (p:PageLayout)-[:HAS_SECTION]->()-[:HAS_ENTITY]->(e:EntityLayout)
WITH p, e ORDER BY CASE WHEN e.bbox_center IS NULL THEN 2.0 ELSE e.bbox_center[1] END,
                   CASE WHEN e.bbox_center IS NULL THEN 2.0 ELSE e.bbox_center[0] END
WITH p, collect(e) AS entities
UNWIND range(0, size(entities) - 1) AS i
WITH entities[i] AS e, i
SET e.order = CASE WHEN e.bbox_center IS NULL THEN null ELSE i + 1 END
"""

CONSOLIDATE_TABLES = """
MATCH (p:PageLayout)-[:HAS_TABLE]->(t:TableLayout)
WITH p, t, [i IN range(0, size(t.obs_x) - 1) WHERE t.obs_x[i] >= 0 | i] AS idx
CALL (t, idx) {
  UNWIND (CASE WHEN size(idx) = 0 THEN [null] ELSE idx END) AS i
  RETURN percentileCont(t.obs_x[i], 0.5) AS x, percentileCont(t.obs_y[i], 0.5) AS y,
         percentileCont(t.obs_w[i], 0.5) AS w, percentileCont(t.obs_h[i], 0.5) AS h
}
SET t.presence_rate = round(toFloat(size(t.obs_doc_ids)) / p.doc_count, 3),
    t.mean_rows = round(toFloat(reduce(s = 0, r IN t.obs_rows | s + r)) / size(t.obs_rows), 1),
    t.region_bbox = CASE WHEN x IS NULL THEN null ELSE [round(x, 4), round(y, 4), round(w, 4), round(h, 4)] END
WITH t
MATCH (t)-[:HAS_COLUMN]->(c:ColumnLayout)
WITH c, [x IN c.obs_x WHERE x >= 0] AS xs
CALL (xs) {
  UNWIND (CASE WHEN size(xs) = 0 THEN [null] ELSE xs END) AS x
  RETURN percentileCont(x, 0.1) AS lo, percentileCont(x, 0.9) AS hi
}
SET c.fill_rate = round(toFloat(size([f IN c.obs_filled WHERE f])) / size(c.obs_filled), 3),
    c.x_band = CASE WHEN lo IS NULL THEN null ELSE [round(lo, 4), round(hi, 4)] END
"""


def rule_prior(strength: float) -> tuple[float, float, float, int]:
    """Empirical-Bayes Beta prior for p_in_range: (alpha, beta, p0, layouts used). p0 is the out-of-sample hit rate
    of a one-document rule: over ordered pairs of boxes of one vendor layout, how often centre j lies within
    BOX_RANGE_MIN of centre i. Averaged per layout, then over layouts."""
    rates = []
    for r in cypher("""MATCH (v:Vendor)-[:HAS_LAYOUT]->()-[:HAS_SECTION]->()-[:HAS_ENTITY]->(e:EntityLayout)
                       WHERE v.key <> $all AND NOT e.key IN $defaulted
                       RETURN e.obs_x AS x, e.obs_y AS y, e.obs_w AS w, e.obs_h AS h""",
                    all=ALL_VENDORS, defaulted=sorted(H.DEFAULTED_KEYS)):
        centres = [(x + w / 2, y + h / 2) for x, y, w, h in zip(r["x"], r["y"], r["w"], r["h"]) if x >= 0]
        if len(centres) >= 2:
            rates.append(np.mean([abs(a[0] - b[0]) <= BOX_RANGE_MIN and abs(a[1] - b[1]) <= BOX_RANGE_MIN
                                  for i, a in enumerate(centres) for j, b in enumerate(centres) if i != j]))
    p0 = float(np.mean(rates))
    return strength * p0, strength * (1 - p0), p0, len(rates)


def consolidate() -> None:
    cypher(CONSOLIDATE_ENTITIES, tol=BBOX_TOLERANCE, min_fraction=MIN_SUPPORT_FRACTION, min_docs=MIN_SUPPORT_DOCS,
           strong_docs=STRONG_SUPPORT_DOCS)
    cypher(CONSOLIDATE_RANGES, k=BOX_RANGE_SPREAD_K, min_margin=BOX_RANGE_MIN, max_margin=BOX_RANGE_MAX,
           alpha=RULE_PRIOR[0], beta=RULE_PRIOR[1])
    cypher(ORDER_ENTITIES)
    cypher(CONSOLIDATE_TABLES)


*RULE_PRIOR, P0, n_prior_layouts = rule_prior(RULE_PRIOR_STRENGTH)
print(f"Empirical prior: a one-document rule catches the next document {P0:.3f} of the time "
      f"({n_prior_layouts} entity layouts with 2+ boxes) -> Beta({RULE_PRIOR[0]:.2f}, {RULE_PRIOR[1]:.2f})")
consolidate()

rules_df = cypher_df("""
MATCH (v:Vendor)-[:HAS_LAYOUT]->(p:PageLayout)-[:HAS_SECTION]->()-[:HAS_ENTITY]->(e:EntityLayout)
WHERE v.key <> $all AND NOT e.key IN $defaulted
RETURN v.key AS vendor, p.id AS layout, p.doc_count AS docs, e.key AS entity, e.order AS order,
       e.fill_rate AS fill_rate, e.n_boxes AS n_boxes, e.support AS support, e.n_in_range AS n_in_range,
       e.p_in_range AS p_in_range, e.bbox_range AS bbox_range, e.top_labels AS top_labels, e.strength AS strength""",
                     all=ALL_VENDORS, defaulted=sorted(H.DEFAULTED_KEYS))
print("v2 strength labels against the new p_in_range (vendor trees, entities with a range):")
display(rules_df.dropna(subset=["p_in_range"]).groupby("strength").p_in_range.describe()[["count", "mean", "min", "max"]])
EXAMPLE_VENDOR = vendors_df.sort_values("n_train", ascending=False).vendor_key.iloc[0]
rules_df[(rules_df.vendor == EXAMPLE_VENDOR) & rules_df.p_in_range.notna()].sort_values(["layout", "order"]).head(20)

## Pass-1 guide: ungated, by cosine similarity
`memory_guide(vendor_name, page_number, structure)` picks the paths that guide one page. It is v2's guide, except
that the vector compared is the **structure vector** (fill + position bins), not the fill vector:

1. **Known vendor.** Take that vendor's PageLayout for the page number whose structure vector is closest (cosine).
2. **Unknown vendor, or no layout for that page.** Take the `GUIDE_NEIGHBOURS` closest PageLayouts of any vendor.
3. **Always** add the `ALL_VENDORS` layout for corpus-wide label habits.

Every rule of the chosen layouts goes to Gemini, whatever its strength. The prompt marks strength but doesn't
filter on it.

In [ ]:
def layout_rules(layout_id: str) -> pd.DataFrame:
    """Every rule-bearing entity path and table path of one PageLayout, in JSON order."""
    entities = cypher_df("""
        MATCH (v:Vendor)-[:HAS_LAYOUT]->(p:PageLayout {id: $id})-[hs:HAS_SECTION]->(s)-[he:HAS_ENTITY]->(e)
        WHERE (e.strength <> 'none' OR size(e.top_labels) > 0) AND NOT e.key IN $defaulted
        RETURN v.key AS vendor, p.id AS layout_id, p.doc_count AS layout_docs, 'entity' AS kind, s.name AS section,
               e.key AS name, hs.json_position AS sec_pos, he.json_position AS pos, e.order AS order,
               e.fill_rate AS fill_rate, e.support AS support, e.n_boxes AS n_boxes, e.strength AS strength,
               e.bbox_median AS bbox, e.top_labels AS top_labels
        ORDER BY sec_pos, pos""", id=layout_id, defaulted=sorted(H.DEFAULTED_KEYS))
    tables = cypher_df("""
        MATCH (v:Vendor)-[:HAS_LAYOUT]->(p:PageLayout {id: $id})-[:HAS_TABLE]->(t)
        OPTIONAL MATCH (t)-[hc:HAS_COLUMN]->(c) WHERE c.fill_rate >= 0.5
        WITH v, p, t, c ORDER BY hc.json_position
        RETURN v.key AS vendor, p.id AS layout_id, p.doc_count AS layout_docs, 'table' AS kind,
               'Tables' AS section, t.type AS name, 99 AS sec_pos, 0 AS pos, null AS order,
               t.presence_rate AS fill_rate, size(t.obs_doc_ids) AS support, null AS n_boxes,
               CASE WHEN t.presence_rate >= 0.5 THEN 'strong' ELSE 'weak' END AS strength,
               t.region_bbox AS bbox, collect(c.header) AS top_labels""", id=layout_id)
    rules = pd.concat([entities, tables], ignore_index=True)
    if not rules.empty:   # keep list columns as lists (concat can turn missing values into NaN)
        rules["bbox"] = rules["bbox"].map(lambda b: b if isinstance(b, list) else None)
        rules["top_labels"] = rules["top_labels"].map(lambda l: l if isinstance(l, list) else [])
    return rules


NEAREST_LAYOUTS = """
MATCH (v:Vendor)-[:HAS_LAYOUT]->(p:PageLayout {page_number: $page})
WHERE v.key <> $all AND ($vendor IS NULL OR v.key = $vendor)
RETURN p.id AS layout_id, v.key AS vendor, p.doc_count AS docs,
       vector.similarity.cosine(p.structure_vector, $structure) AS structure_similarity
ORDER BY structure_similarity DESC, docs DESC LIMIT $k
"""


def memory_guide(vendor_name: str, page_number: int, structure: list[float], k: int = GUIDE_NEIGHBOURS) -> dict:
    """The paths that should guide one page, and how they were chosen."""
    vkey = H.vendor_key(vendor_name)
    page = min(page_number, 2)  # the corpus has at most two pages; later pages reuse page-2 layouts
    own = cypher(NEAREST_LAYOUTS, page=page, vendor=vkey, all=ALL_VENDORS, structure=structure, k=1)
    if own:
        chosen = [{**own[0], "source": "same vendor"}]
    else:
        chosen = [{**n, "source": "similar structure, other vendor"}
                  for n in cypher(NEAREST_LAYOUTS, page=page, vendor=None, all=ALL_VENDORS, structure=structure, k=k)]
    chosen.append({"layout_id": f"{ALL_VENDORS}|p{page}|v0", "vendor": ALL_VENDORS, "source": "corpus labels",
                   "structure_similarity": None})
    return {"vendor_key": vkey, "vendor_known": bool(own), "page": page_number, "layouts": chosen}


def ungated_guides(evidence: dict) -> list[dict]:
    """One v2 guide per page of an unguided extraction (its vendor name and per-page structure vectors)."""
    return [memory_guide(evidence["vendor_name"], page["number"],
                         H.structure_vector(page, POSITION_GRID, POSITION_WEIGHT)) for page in evidence["pages"]]


def rules_to_text(guide: dict) -> str:
    """Render an ungated guide's layouts as prompt text: one line per entity rule (box + label), one per table."""
    lines = [f"PAGE {guide['page']}"]
    for layout in guide["layouts"]:
        rules = layout_rules(layout["layout_id"])
        if rules.empty:
            continue
        sim = layout.get("structure_similarity")
        lines.append(f"  Layout (vendor tree '{layout['vendor']}', {layout['source']}"
                     + (f", structure similarity {sim:.2f}" if sim is not None else "")
                     + f", built from {rules.layout_docs.iloc[0]} invoice(s)):")
        corpus = layout["vendor"] == ALL_VENDORS
        for r in rules.itertuples():
            labels = f" labelled {', '.join(repr(l) for l in r.top_labels)}" if r.top_labels else ""
            if r.kind == "table":
                where = f" in region box_2d {H.bbox_to_box2d(r.bbox)}" if r.bbox and not corpus else ""
                lines.append(f"    TABLE {r.name}: present on {r.fill_rate:.0%} of pages{where}; usually filled "
                             f"columns: {', '.join(r.top_labels) or '-'}")
            elif corpus:
                if r.top_labels:
                    lines.append(f"    {r.name}: common labels {', '.join(repr(l) for l in r.top_labels)}")
            else:
                where = (f"value usually at box_2d {H.bbox_to_box2d(r.bbox)} ({r.strength}: {r.support} of "
                         f"{r.n_boxes} located)" if r.bbox and r.strength != "none" else "position varies")
                lines.append(f"    {r.name}: filled on {r.fill_rate:.0%} of pages; {where}{labels}")
    return "\n".join(lines)


example_doc = next(d for d in sorted(test_ids) if train_count[records[d]["vendor_key"]] > 0)
print("Ungated guide built from the ground truth of", example_doc, "(before any Gemini call):\n")
print(rules_to_text(ungated_guides(records[example_doc])[0]))

## The posterior over layouts
Before consolidation, every training page was put into exactly one group: its vendor's PageLayout for that page
number and variant. For a new page we observe evidence $e$ from the unguided extraction and ask how probable each
group is. The group might also be one the memory has never seen.

**Candidates.** Every vendor PageLayout $L$ with the same page number (pages ≥ 2 use page-2 layouts), plus a
class **NEW**: "this page comes from a vendor or layout that isn't in memory". NEW has no rules of its own. It
behaves like the corpus as a whole, the `ALL_VENDORS` tree.

**Evidence** $e = (\hat k, x, c)$:

* $\hat k$: the vendor key of the extracted Vendor Name
* $x \in \{0,1\}^{20}$: which of the 18 scorable entities are filled and which of the 2 tables are present
* $c = \{c_k\}$: the box centre of every filled entity that has a box

**Bayes' rule.** Assume the three pieces of evidence are conditionally independent given the layout:

$$P(L \mid e) \;=\; \frac{P(L)\; P(\hat k \mid L)\; P(x \mid L)^{1/T}\; p(c \mid x, L)}
{\sum_{L'} P(L')\; P(\hat k \mid L')\; P(x \mid L')^{1/T}\; p(c \mid x, L')}$$

The denominator is the same for all $L$, so the code adds up log-terms per candidate and normalizes once with
log-sum-exp (`helpers.normalize_log`). The four factors follow.

### 1. Prior $P(L)$
Before looking at the page, how likely is each group?

* **NEW, by Good–Turing.** The share of training documents whose vendor appears exactly once estimates the
  probability that the next document comes from a vendor not yet seen:
  $P(\text{NEW}) = N_1 / N$.
* **Known layouts.** The remaining mass is split by how many training pages with this page number each layout
  holds: $P(L) = (1 - P(\text{NEW})) \cdot n_L / \sum_{L'} n_{L'}$.

### 2. Name $P(\hat k \mid L)$: a noisy channel
The extracted vendor name is strong evidence, but it is fallible: v2's unguided Vendor Name was exact on only 78%
of pages. Model the extraction as a channel that reports the right vendor key with probability $1 - \varepsilon$
(`NAME_ERROR`). Otherwise it reports some other key, spread evenly over the $M$ known keys:

$$P(\hat k \mid L) = \begin{cases} 1 - \varepsilon & \hat k = \text{vendor}(L) \\
\varepsilon\,\rho & \hat k \text{ is a variant of vendor}(L) \\
\varepsilon (1 - \rho) / M & \text{otherwise} \end{cases}
\qquad
P(\hat k \mid \text{NEW}) = \begin{cases} 1 - \varepsilon & \hat k \text{ is not a known key} \\ \varepsilon / M & \text{otherwise} \end{cases}$$

**Variants.** Errors are not random. Most often the name comes back as a *variant* of the true one: a "dba"
suffix, an extra word, a dropped word. On the first run, "konoike pacific california dba kpac kpac general kpac
e street" was scored as a complete stranger to the "konoike pacific california" tree. So a share $\rho$
(`NAME_VARIANT`) of the error mass goes to variants of the true key, and the rest is spread over all keys.
`helpers.keys_related` calls two keys variants when one's words start the other's, or when they share at least
half their words. A variant reading then multiplies a layout's odds by $\rho M/(1-\rho) \approx 100$ over an
unrelated key, against about 1000 for an exact match. Related companies with different names (amerisourcebergen /
cencora) are not variants by this test; that would need an alias table.

With $\varepsilon = 0.1$ and about 110 known keys, a matching name multiplies the odds by about 1000, but a
strong mismatch in the fields and boxes can still overturn it. An empty name ($\hat k$ = `UNCLASSIFIED`) is
treated as no evidence.

### 3. Fields $P(x \mid L)$: Bernoulli naive Bayes
These are the "term frequencies" of each group. Field $f$ is filled on $n_{L,f}$ of the $n_L$ training pages of
$L$. A field appears at most once per page, so the right event model is **Bernoulli** (present or absent), not
multinomial. Absences count as evidence too: a page with a PO Number counts against a layout that never prints
one.

* **Shrinkage.** Layouts built from one or two pages would give rates of exactly 0 or 1, and then a single
  disagreement would make $P(x \mid L) = 0$. So each layout's rate is smoothed toward the corpus rate
  $\theta_{\text{ALL},f}$ with $m$ pseudo-pages (`FIELD_SMOOTHING_M`):
  $$\theta_{L,f} = \frac{n_{L,f} + m\,\theta_{\text{ALL},f}}{n_L + m}$$
  This is the mean of a Beta prior centred on the corpus. NEW uses $\theta_{\text{ALL}}$ itself.
* **Extraction channel.** $x$ comes from Gemini, not the ground truth. A printed field is missed with probability
  $r$ (`FIELD_MISS`), and an absent field is filled with probability $f$ (`FIELD_FALSE_FILL`), so
  $$P(\hat x_f = 1 \mid L) = \theta_{L,f}(1 - r) + (1 - \theta_{L,f})\,f$$
* **Naive Bayes over fields.**
  $$\log P(x \mid L) = \sum_f \hat x_f \log P(\hat x_f = 1 \mid L) + (1 - \hat x_f) \log P(\hat x_f = 0 \mid L)$$
* **Temperature.** Fields are not independent (Bank Account, Routing Number and Bank Account Number come
  together), so naive Bayes counts the same evidence several times and is overconfident. Raising the term to
  $1/T$ (`FIELD_TEMPERATURE`) discounts it. Part 2 of the evaluation sweeps $T$.

### 4. Boxes $p(c \mid x, L)$: does the page put values where $L$ does?
For each extracted centre $c_k$, layout $L$ either has a range $R_{L,k}$ for entity $k$ or it doesn't. Recall
$q_{L,k}$ = `p_in_range`, the probability that a centre falls in the range. The natural density puts mass
$q_{L,k}$ uniformly inside the range and $1 - q_{L,k}$ uniformly over the rest of the page:

$$p(c_k \mid L) = \begin{cases} q_{L,k} / |R_{L,k}| & c_k \in R_{L,k} \\ (1 - q_{L,k}) / (1 - |R_{L,k}|) & \text{otherwise} \end{cases}$$

$|R|$ is the range's area in page fractions. With no range (and always for NEW), $p(c_k \mid L) = 1$, the uniform
density on the unit page. This term has to be a **density**, not a probability. A layout with a narrow range
earns a large reward ($q/|R|$) when the centre lands in it. A layout with a wide range earns little. A layout
that says nothing about $k$ is neutral. Without dividing by the area, a layout could win just by having huge
ranges. This is the positional evidence that can tell apart two vendors who print the same fields in different
places.

In [ ]:
FEATURES = [k for k in H.KV_KEYS if k not in H.DEFAULTED_KEYS] + list(H.TABLE_TYPES)   # the 20 bits of x
BOX_KEYS = [k for k in H.KV_KEYS if k not in H.DEFAULTED_KEYS]
NEW = "NEW"
TERMS = ("prior", "name", "fields", "boxes")

ENTITY_STATS = """
MATCH (v:Vendor)-[:HAS_LAYOUT]->(p:PageLayout)-[:HAS_SECTION]->()-[:HAS_ENTITY]->(e:EntityLayout)
RETURN v.key AS vendor, p.id AS layout_id, p.page_number AS page, p.doc_count AS docs, e.key AS key,
       e.n_filled AS n_filled, e.bbox_range AS bbox_range, e.p_in_range AS q, e.bbox_median AS bbox_median,
       e.strength AS strength, e.top_labels AS top_labels
"""
TABLE_STATS = """
MATCH (p:PageLayout)-[:HAS_TABLE]->(t:TableLayout)
RETURN p.id AS layout_id, t.type AS key, size(t.obs_doc_ids) AS n_filled
"""


def load_layouts() -> dict[str, dict]:
    """Every PageLayout's counts and rules, pulled out of the graph once: the posterior is computed in numpy."""
    layouts = {}
    for r in cypher(ENTITY_STATS):
        L = layouts.setdefault(r["layout_id"], {"id": r["layout_id"], "vendor": r["vendor"], "page": r["page"],
                                                "docs": r["docs"], "n_filled": {}, "rules": {}, "labels": {}})
        L["n_filled"][r["key"]] = r["n_filled"]
        L["labels"][r["key"]] = r["top_labels"]
        if r["bbox_range"] and r["key"] in BOX_KEYS:
            L["rules"][r["key"]] = {"range": r["bbox_range"], "q": r["q"], "bbox_median": r["bbox_median"],
                                    "strength": r["strength"]}
    for r in cypher(TABLE_STATS):
        layouts[r["layout_id"]]["n_filled"][r["key"]] = r["n_filled"]
    for L in layouts.values():
        L["n_filled"] = np.array([L["n_filled"].get(f, 0) for f in FEATURES], dtype=float)
    return layouts


LAYOUTS = load_layouts()
VENDOR_LAYOUTS = {i: L for i, L in LAYOUTS.items() if L["vendor"] != ALL_VENDORS}
CORPUS = {L["page"]: L for L in LAYOUTS.values() if L["vendor"] == ALL_VENDORS}
THETA_ALL = {page: L["n_filled"] / L["docs"] for page, L in CORPUS.items()}
KNOWN_KEYS = {L["vendor"] for L in VENDOR_LAYOUTS.values()}
P_NEW = sum(1 for n in train_count.values() if n == 1) / len(train_ids)       # Good-Turing N1 / N
print(f"{len(VENDOR_LAYOUTS)} vendor layouts, {len(KNOWN_KEYS)} known vendor keys, P(NEW) = {P_NEW:.3f}")


def presence(page: dict) -> np.ndarray:
    """x: 1 for every filled entity and every table present on the page, in FEATURES order."""
    filled = {k for k, f in page["fields"].items() if f["value"]} | {t["type"] for t in page["tables"] if t["rows"]}
    return np.array([float(f in filled) for f in FEATURES])


def theta(L: dict | str, page: int, m: float = FIELD_SMOOTHING_M) -> np.ndarray:
    """Smoothed P(feature present | L): the layout's counts shrunk toward the corpus with m pseudo-pages."""
    if L == NEW:
        return THETA_ALL[page]
    return (L["n_filled"] + m * THETA_ALL[page]) / (L["docs"] + m)


def layout_scores(page: dict, vendor_name: str, m: float = FIELD_SMOOTHING_M,
                  T: float = FIELD_TEMPERATURE) -> pd.DataFrame:
    """One row per candidate layout (and NEW): the log of each factor of Bayes' rule, before normalization."""
    pn = min(page["number"], 2)
    candidates = [L for L in VENDOR_LAYOUTS.values() if L["page"] == pn]
    pages_here = sum(L["docs"] for L in candidates)
    x = presence(page)
    k_hat = H.vendor_key(vendor_name)
    centres = {k: H.bbox_center(f["bbox"]) for k, f in page["fields"].items() if f["bbox"] and k in BOX_KEYS}
    rows = []
    for L in candidates + [NEW]:
        is_new = L == NEW
        # 1. prior: Good-Turing mass for NEW, the rest by training pages per layout
        prior = P_NEW if is_new else (1 - P_NEW) * L["docs"] / pages_here
        # 2. name: noisy channel over vendor keys (exact, variant spelling, unrelated); an empty name is no evidence
        if k_hat == H.UNCLASSIFIED:
            name = 0.0
        elif is_new:
            name = math.log(1 - NAME_ERROR) if k_hat not in KNOWN_KEYS else math.log(NAME_ERROR / len(KNOWN_KEYS))
        elif k_hat == L["vendor"]:
            name = math.log(1 - NAME_ERROR)
        elif H.keys_related(k_hat, L["vendor"]):
            name = math.log(NAME_ERROR * NAME_VARIANT)
        else:
            name = math.log(NAME_ERROR * (1 - NAME_VARIANT) / len(KNOWN_KEYS))
        # 3. fields: Bernoulli naive Bayes through the extraction channel, tempered by 1/T
        p1 = theta(L, pn, m) * (1 - FIELD_MISS) + (1 - theta(L, pn, m)) * FIELD_FALSE_FILL
        fields = float(np.sum(x * np.log(p1) + (1 - x) * np.log(1 - p1))) / T
        # 4. boxes: piecewise-uniform density around each of L's ranges; 1 (log 0) where L has no range
        boxes = 0.0
        for k, c in centres.items():
            rule = None if is_new else L["rules"].get(k)
            if rule:
                area = max(H.range_area(rule["range"]), 1e-6)
                boxes += math.log(rule["q"] / area) if H.in_range(c, rule["range"]) else \
                         math.log((1 - rule["q"]) / (1 - area))
        rows.append({"layout_id": NEW if is_new else L["id"], "vendor": NEW if is_new else L["vendor"],
                     "prior": math.log(prior), "name": name, "fields": fields, "boxes": boxes})
    return pd.DataFrame(rows).set_index("layout_id")


def posterior(scores: pd.DataFrame, terms: tuple[str, ...] = TERMS) -> dict[str, float]:
    """P(L | e) over the candidates, using only the chosen factors (for ablation)."""
    return H.normalize_log(scores[list(terms)].sum(axis=1).to_dict())


def vendor_posterior(scores: pd.DataFrame, post: dict[str, float]) -> dict[str, float]:
    """P(vendor | e): the posterior summed over each vendor's layouts."""
    out = defaultdict(float)
    for layout_id, p in post.items():
        out[scores.at[layout_id, "vendor"]] += p
    return dict(out)


example_page = records[example_doc]["pages"][0]
example_scores = layout_scores(example_page, records[example_doc]["vendor_name"])
example_scores["posterior"] = pd.Series(posterior(example_scores))
print(f"Top candidates for {example_doc} page 1 (true vendor: {records[example_doc]['vendor_key']}), "
      "ground-truth evidence:")
example_scores.sort_values("posterior", ascending=False).head(6).round(3)

**Checks.**

* **Normalization.** Every posterior sums to 1.
* **Shrinkage limit.** With $m \to \infty$, every $\theta_L$ collapses to $\theta_{\text{ALL}}$, so the field term
  is identical for all candidates and cancels in Bayes' rule. With only the prior and fields switched on, the
  posterior must then equal the prior.
* **In-sample recovery.** A training page of a strong-tree vendor, scored with its own ground truth, should have
  its own vendor as the MAP.

In [ ]:
check_page, check_name = records[example_doc]["pages"][0], records[example_doc]["vendor_name"]
assert abs(sum(posterior(example_scores).values()) - 1) < 1e-9
flat = layout_scores(check_page, check_name, m=1e12)
assert np.allclose(pd.Series(posterior(flat, ("prior", "fields"))), pd.Series(posterior(flat, ("prior",)))), \
    "with infinite smoothing the field term must cancel"

strong_vendors = {v for v, n in train_count.items() if n >= STRONG_SUPPORT_DOCS and v != H.UNCLASSIFIED}
recovered = []
for doc_id in sorted(d for d in train_ids if records[d]["vendor_key"] in strong_vendors):
    for page in records[doc_id]["pages"]:
        s = layout_scores(page, records[doc_id]["vendor_name"])
        vp = vendor_posterior(s, posterior(s))
        recovered.append(max(vp, key=vp.get) == records[doc_id]["vendor_key"])
print(f"In-sample: {np.mean(recovered):.1%} of {len(recovered)} training pages of strong-tree vendors "
      "put their own vendor at the MAP")
assert np.mean(recovered) >= 0.9

## From the layout posterior to a confidence for each rule
The gated guide shows the rules of one layout: the MAP layout $L^* = \arg\max_L P(L \mid e)$. The confidence of
its rule for entity $k$ is the probability that the value really is inside the rule's range, given everything
observed about the page. By the **law of total probability** over which layout the page really comes from:

$$\text{conf}_k \;=\; P(c_k \in R_{L^*,k} \mid e) \;=\; \sum_{L} P(L \mid e)\; P(c_k \in R_{L^*,k} \mid L)$$

Each term asks: if the page really is $L$, how likely is the value to sit inside the range we're showing?

| True layout $L$ | $P(c_k \in R_{L^*,k} \mid L)$ |
|---|---|
| $L = L^*$ | $q_{L^*,k}$, the rule's own reliability |
| another layout with a range for $k$ | the mass its piecewise-uniform density puts on $R^* = R_{L^*,k}$: $\; q_L \frac{\lvert R_L \cap R^*\rvert}{\lvert R_L\rvert} + (1 - q_L)\frac{\lvert R^*\rvert - \lvert R_L \cap R^*\rvert}{1 - \lvert R_L\rvert}$ |
| another layout without a range for $k$, or NEW | $\lvert R^* \rvert$, the chance a uniformly placed value lands in $R^*$ |

This is why the confidence isn't just $P(L^* \mid e)$. If the runner-up layout puts the value in the same place,
the rule is right either way and the confidence stays high. If the page is probably a NEW layout, the confidence
falls toward the area of the range, around 0.01–0.09. It also isn't just $q_{L^*,k}$: a very reliable rule of the
wrong layout is still wrong.

The guide also reports how likely the field is to be printed at all, by the same total-probability step:
$P(\text{filled}_k \mid e) = \sum_L P(L \mid e)\, \theta_{L,k}$.

**The gate** turns the confidence into a decision:

| Confidence | What Gemini sees |
|---|---|
| ≥ `GATE_BOX` | the rule's typical box (`bbox_median` as `box_2d`) and its confidence |
| `GATE_LABEL` to `GATE_BOX` | label hint only, marked "position uncertain" |
| < `GATE_LABEL` | nothing: the rule is **withheld** |

**Ranges score, point boxes guide.** The range is what the confidence is computed on, because it tolerates
noisy boxes. Gemini is shown the rule's point box, as in the ungated pass. On the first run, showing the range
itself as a "search region" made Gemini draw its own, much looser boxes: `bbox_overlap` fell from 0.72 to 0.26 on
the same correct strong rules. The gated pass should differ from the ungated one only in *which* rules it shows
and the confidence attached, not in how a rule is presented.

Entities that $L^*$ never located have no range, so they get no confidence. They keep a label hint when they
have one. When NEW is the MAP, no vendor rules are shown, only the corpus-wide labels.

In [ ]:
def rule_confidence(post: dict[str, float], key: str, chosen: str) -> float | None:
    """P(c_k in R_{L*,k} | e) = sum_L P(L | e) P(c_k in R_{L*,k} | L), by the law of total probability."""
    rule = VENDOR_LAYOUTS[chosen]["rules"].get(key)
    if rule is None:
        return None
    r_star, a_star = rule["range"], H.range_area(rule["range"])
    conf = 0.0
    for layout_id, p in post.items():
        other = None if layout_id in (NEW, chosen) else VENDOR_LAYOUTS[layout_id]["rules"].get(key)
        if layout_id == chosen:
            hit = rule["q"]
        elif other is None:
            hit = a_star
        else:
            a, inter = H.range_area(other["range"]), H.range_overlap(other["range"], r_star)
            hit = other["q"] * inter / a + (1 - other["q"]) * (a_star - inter) / (1 - a)
        conf += p * hit
    return conf


def gate_of(conf: float | None) -> str:
    if conf is None:
        return "no_position"
    return "box" if conf >= GATE_BOX else "label" if conf >= GATE_LABEL else "withheld"


def page_rules(page: dict, vendor_name: str, T: float = FIELD_TEMPERATURE) -> dict:
    """The gated guide for one page: posterior summary, the MAP layout, and every rule of it with its confidence
    and gate decision (withheld rules included, so they can be evaluated)."""
    pn = min(page["number"], 2)
    scores = layout_scores(page, vendor_name, T=T)
    post = posterior(scores)
    chosen = max(post, key=post.get)
    p_filled = sum(p * theta(NEW if i == NEW else VENDOR_LAYOUTS[i], pn) for i, p in post.items())
    out = {"page": page["number"], "vendor_key": H.vendor_key(vendor_name), "chosen": chosen,
           "chosen_vendor": scores.at[chosen, "vendor"], "p_chosen": post[chosen], "p_new": post[NEW],
           "top_layouts": dict(sorted(post.items(), key=lambda kv: -kv[1])[:5]), "rules": [], "tables": []}
    if chosen == NEW:
        return out
    L = VENDOR_LAYOUTS[chosen]
    out["chosen_docs"] = L["docs"]
    for key in BOX_KEYS:
        rule = L["rules"].get(key, {})
        conf = rule_confidence(post, key, chosen)
        out["rules"].append({"name": key, "confidence": conf, "gate": gate_of(conf),
                             "p_filled": float(p_filled[FEATURES.index(key)]), "range": rule.get("range"),
                             "bbox_median": rule.get("bbox_median"), "q": rule.get("q"),
                             "strength": rule.get("strength", "none"), "labels": L["labels"].get(key) or []})
    for table in H.TABLE_TYPES:
        out["tables"].append({"name": table, "p_present": float(p_filled[FEATURES.index(table)])})
    return out


def gated_guides(evidence: dict) -> list[dict]:
    return [page_rules(page, evidence["vendor_name"]) for page in evidence["pages"]]


def rules_to_text_gated(guide: dict) -> str:
    """Render a gated guide: only rules that passed the gate, each with its probabilities."""
    lines = [f"PAGE {guide['page']}"]
    if guide["chosen"] == NEW:
        lines.append(f"  No remembered layout fits this page (P(unseen layout) = {guide['p_new']:.2f}); no "
                     "vendor-specific rules.")
    else:
        lines.append(f"  Layout from vendor tree '{guide['chosen_vendor']}' (P(this layout | page) = "
                     f"{guide['p_chosen']:.2f}, built from {guide['chosen_docs']} invoice(s)):")
        for r in guide["rules"]:
            labels = f"; labelled {', '.join(repr(l) for l in r['labels'])}" if r["labels"] else ""
            if r["gate"] == "box":
                where = f"value usually at box_2d {H.bbox_to_box2d(r['bbox_median'])} " \
                        f"(confidence {r['confidence']:.2f})"
            elif r["gate"] == "label":
                where = f"position uncertain (confidence {r['confidence']:.2f})"
            elif r["gate"] == "no_position" and labels:
                where = "position not remembered"
            else:
                continue
            lines.append(f"    {r['name']}: expected with probability {r['p_filled']:.2f}; {where}{labels}")
        columns = {r.name: r.top_labels for r in layout_rules(guide["chosen"]).itertuples() if r.kind == "table"}
        for t in guide["tables"]:
            if t["p_present"] >= 0.5:
                lines.append(f"    TABLE {t['name']}: present with probability {t['p_present']:.2f}; usually "
                             f"filled columns: {', '.join(columns.get(t['name'], [])) or '-'}")
    corpus = LAYOUTS[f"{ALL_VENDORS}|p{min(guide['page'], 2)}|v0"]["labels"]
    lines.append("  Labels common across all vendors:")
    lines += [f"    {k}: {', '.join(repr(l) for l in corpus[k])}" for k in BOX_KEYS if corpus.get(k)]
    return "\n".join(lines)


example_gated = page_rules(example_page, records[example_doc]["vendor_name"])
assert all(0 <= r["confidence"] <= 1 for r in example_gated["rules"] if r["confidence"] is not None)
display(pd.DataFrame(example_gated["rules"])[["name", "p_filled", "q", "confidence", "gate", "strength"]].round(3))
print(rules_to_text_gated(example_gated))

## Test run on the held-out third
For each test PDF:

1. **Call 0, unguided.** Gemini extracts values and `box_2d` for every entity and table with no memory. Its vendor
   name, fill vector and boxes are the **evidence** for both guides. It is saved to `results/unguided/` and not
   scored.
2. **Pass 1, ungated, guided.** The cosine guide (`ungated_guides`) is built from call 0 and added to the system
   prompt. Saved to `results/pass1/`, with its guide in `results/guides/pass1/`.
3. **Pass 2, gated.** The confidence-gated guide (`gated_guides`) is built from the same call 0. Saved to
   `results/pass2/` and `results/guides/pass2/`.

Each output is converted back into the full output schema (`helpers.compact_to_envelope`), so it can be diffed
against ground truth and against the other pass. Guides are cheap, so they are always rebuilt from the current
graph. Call 0 and pass 1 outputs are reused whenever they exist: pass 1 is the frozen ungated baseline. A pass-2
output is reused only while its guide's prompt text is unchanged. Edit the posterior, the gate or the rendering,
and only the documents whose gated prompt changed go back to Gemini.

In [ ]:
from google import genai

client = genai.Client(vertexai=True, project=GCP_PROJECT, location=GCP_LOCATION)
SYSTEM_PROMPT = H.EXTRACT_PROMPT.format(catalog=H.catalog_text())
SCHEMA = H.extraction_schema()


def save_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=1))


def run_unguided(doc_id: str, pdf_path: Path, facts: dict) -> dict:
    """Call 0: the evidence for both guides, as a flat record."""
    path = RESULTS_DIR / "unguided" / f"{doc_id}.json"
    if not path.exists():
        pred = H.gemini_extract(client, OCR_MODEL, pdf_path, SYSTEM_PROMPT, SCHEMA)
        save_json(path, H.compact_to_envelope(pred, facts, OCR_MODEL))
    return H.flatten_document(doc_id, json.loads(path.read_text()), facts)


def run_guided(doc_id: str, pdf_path: Path, facts: dict, evidence: dict, name: str, guides_fn, render_fn,
               suffix: str, refresh: bool) -> None:
    """One guided pass: guide from the evidence, rules appended to the system prompt, output saved to <name>/.
    A saved output is reused if it exists, unless `refresh` is set and the guide's prompt text has changed since
    it was produced."""
    guides = guides_fn(evidence)
    rules = "\n\n".join(render_fn(g) for g in guides)
    guide_path, out = RESULTS_DIR / "guides" / name / f"{doc_id}.json", RESULTS_DIR / name / f"{doc_id}.json"
    stale = refresh and guide_path.exists() and json.loads(guide_path.read_text())["rules_text"] != rules
    save_json(guide_path, {"guides": guides, "rules_text": rules})
    if stale or not out.exists():
        pred = H.gemini_extract(client, OCR_MODEL, pdf_path, SYSTEM_PROMPT + suffix.format(rules=rules), SCHEMA)
        save_json(out, H.compact_to_envelope(pred, facts, OCR_MODEL))


run_ids = sorted(test_ids)[:LIMIT_TEST_DOCS] if LIMIT_TEST_DOCS else sorted(test_ids)
unguided = {}
for i, doc_id in enumerate(run_ids, 1):
    pdf, facts = INPUT_DIR / f"{doc_id}.pdf", page_facts[doc_id]
    unguided[doc_id] = run_unguided(doc_id, pdf, facts)
    run_guided(doc_id, pdf, facts, unguided[doc_id], "pass1", ungated_guides, rules_to_text, H.GUIDED_SUFFIX,
               refresh=False)                                    # the ungated baseline stays frozen
    run_guided(doc_id, pdf, facts, unguided[doc_id], "pass2", gated_guides, rules_to_text_gated, H.GATED_SUFFIX,
               refresh=True)
    print(f"\r{i}/{len(run_ids)} documents", end="")
print()

## Evaluation
`score_document` compares one saved pass file with its ground truth, one row per (page, entity):

| Column | Meaning |
|---|---|
| `exact` | values equal after whitespace/case normalization |
| `fuzzy` | `difflib` character similarity, 0–1 |
| `bbox_overlap` | intersection over union of the predicted and ground-truth value boxes. Only scored when the ground-truth box is usable; a missing predicted box scores 0 |
| `center_distance` | distance between the two box centres, in page fractions. Summarized as `mean_center_distance` (over predicted boxes) and `center_within_tol`, the share of ground-truth boxes whose predicted centre is within `BBOX_TOLERANCE` on both axes (a missing box counts as a miss) |
| `rule_vs_gt` | `bbox_overlap` of the shown rule's `bbox_median` with ground truth: **was the memory right?** |
| `rule_in_range` | the ground-truth centre lies in the shown rule's range |
| `pred_vs_rule` | `bbox_overlap` of Gemini's box with the rule's `bbox_median`: **did Gemini follow the memory?** |
| `confidence`, `gate` | pass 2 only: the rule's confidence and the gate's decision |

A **shown rule** is a rule that put a position in the prompt. In pass 1 that is the first vendor layout's rule
with strength other than `none`. In pass 2 it is a rule that passed `GATE_BOX`. `Terms Currency` and
ground-truth values marked `valid:false` are excluded. Table cells are not scored here, because the gate acts on
entity rules.

**Why centre metrics too.** Value boxes are tiny (about 1% of the page height), so `bbox_overlap` falls to 0 after a
shift of one box height. The first run showed it mostly measures how exactly Gemini copied a remembered box.
`center_within_tol` asks the question that matters for finding a value: did the box land on it?

In [ ]:
def shown_rules(guide_file: Path) -> dict[int, dict[str, dict]]:
    """Page -> entity -> the rule that put a position in the prompt (bbox_median, range, confidence, gate)."""
    if not guide_file.exists():
        return {}
    out = {}
    for guide in json.loads(guide_file.read_text())["guides"]:
        rules = {}
        if "layouts" in guide:                                   # ungated: first vendor layout with a located rule
            for layout in guide["layouts"]:
                for key, r in VENDOR_LAYOUTS.get(layout["layout_id"], {"rules": {}})["rules"].items():
                    if r["strength"] != "none":
                        rules.setdefault(key, {"bbox": r["bbox_median"], "range": r["range"]})
        else:                                                    # gated: rules that passed GATE_BOX
            rules = {r["name"]: {"bbox": r["bbox_median"], "range": r["range"], "confidence": r["confidence"],
                                 "gate": r["gate"]} for r in guide["rules"] if r["gate"] == "box"}
        out[guide["page"]] = rules
    return out


def score_document(pred: dict, gt: dict, shown: dict) -> list[dict]:
    rows = []
    pred_pages = {p["number"]: p for p in pred["pages"]}
    for gt_page in gt["pages"]:
        pp = pred_pages.get(gt_page["number"], {"fields": {}})
        for key, g in gt_page["fields"].items():
            if key in H.DEFAULTED_KEYS or not g["valid"]:
                continue
            p = pp["fields"].get(key, {"value": "", "bbox": None})
            rule = shown.get(gt_page["number"], {}).get(key)
            rows.append({
                "doc_id": gt["doc_id"], "page": gt_page["number"], "entity": key,
                "gt_filled": bool(g["value"]), "pred_filled": bool(p["value"]),
                "exact": H.normalize_for_match(p["value"]) == H.normalize_for_match(g["value"]),
                "fuzzy": H.string_similarity(p["value"], g["value"]),
                "bbox_overlap": H.bbox_overlap(p["bbox"], g["bbox"]) if g["usable"] else None,
                "center_distance": H.center_distance(p["bbox"], g["bbox"]) if g["usable"] else None,
                "center_within_tol": (bool(p["bbox"]) and H.in_range(H.bbox_center(p["bbox"]), H.box_range(
                    *H.bbox_center(g["bbox"]), BBOX_TOLERANCE))) if g["usable"] else None,
                "rule_vs_gt": H.bbox_overlap(rule["bbox"], g["bbox"]) if g["usable"] and rule else None,
                "rule_in_range": H.in_range(H.bbox_center(g["bbox"]), rule["range"]) if g["usable"] and rule
                                 else None,
                "pred_vs_rule": H.bbox_overlap(p["bbox"], rule["bbox"]) if rule and p["bbox"] else None,
            })
    return rows


def load_pass(directory: Path, doc_id: str, facts: dict) -> dict | None:
    path = directory / f"{doc_id}.json"
    return H.flatten_document(doc_id, json.loads(path.read_text()), facts) if path.exists() else None


def summarize(df: pd.DataFrame) -> pd.Series:
    filled = df[df.gt_filled]
    boxed = df.dropna(subset=["bbox_overlap"])
    return pd.Series({
        "value_exact_acc": filled.exact.mean(),
        "value_fuzzy_sim": filled.fuzzy.mean(),
        "false_fill": (~df.gt_filled & df.pred_filled).sum() / max((~df.gt_filled).sum(), 1),
        "mean_bbox_overlap": boxed.bbox_overlap.mean(),
        "bbox_overlap>=0.5": (boxed.bbox_overlap >= 0.5).mean(),
        "mean_center_distance": pd.to_numeric(boxed.center_distance).mean(),
        "center_within_tol": boxed.center_within_tol.astype(float).mean(),
        "mean_rule_vs_gt": df.rule_vs_gt.mean(),
        "rule_in_range": df.rule_in_range.astype(float).mean(),
        "mean_pred_vs_rule": df.pred_vs_rule.mean(),
    })


def setup_table(df: pd.DataFrame, by: list[str], metrics: list[str]) -> pd.DataFrame:
    """Metrics per group with ungated / gated side by side, plus gated − ungated gains."""
    out = df.groupby(by + ["setup"]).apply(summarize, include_groups=False)[metrics].unstack("setup")
    out.columns = [f"{m}_{s}" for m, s in out.columns]
    for m in metrics:
        if f"{m}_gated" in out and f"{m}_ungated" in out:
            out[f"{m}_gain"] = out[f"{m}_gated"] - out[f"{m}_ungated"]
    return out


# Sanity check: ground truth scored against itself is perfect
self_rows = pd.DataFrame(score_document(records[run_ids[0]], records[run_ids[0]], {}))
assert self_rows.exact.all() and np.allclose(self_rows.bbox_overlap.dropna(), 1.0)

field_rows = []
for doc_id in run_ids:
    for setup, name in [("ungated", "pass1"), ("gated", "pass2")]:
        pred = load_pass(RESULTS_DIR / name, doc_id, page_facts[doc_id])
        if pred is None:
            continue
        shown = shown_rules(RESULTS_DIR / "guides" / name / f"{doc_id}.json")
        field_rows += [{**r, "setup": setup} for r in score_document(pred, records[doc_id], shown)]
fields_df = pd.DataFrame(field_rows)
fields_df["vendor_key"] = fields_df.doc_id.map(lambda d: records[d]["vendor_key"])
fields_df["n_train_docs"] = fields_df.vendor_key.map(lambda v: train_count[v])
fields_df["train_set_size_bucket"] = fields_df.n_train_docs.map(size_bucket)

print("All test documents")
fields_df.groupby("setup").apply(summarize, include_groups=False).round(3)

### Part 1: strong-tree vendors, assuming the rule fits the group
Only test documents whose vendor has at least `STRONG_SUPPORT_DOCS` training documents are used, and only
entities for which that vendor's tree holds a **strong** rule on that page.

**The assumption.** If a strong rule is taken to be right whenever the page really belongs to its layout, then
$P(c_k \in R_{L^*,k} \mid L^*) = 1$. The confidence then reduces to the probability that the page is from that
group:

$$\text{conf}_k = \sum_L P(L \mid e)\, P(c_k \in R_{L^*,k} \mid L) \;\approx\; P(L^* \mid e)$$

So Part 1 is a test of **group identification**: does the posterior put the page in its true vendor's tree, and
are its probabilities honest? Two ablations answer that, both computed without Gemini calls:

* **Which evidence?** Prior only; plus the name; plus fields; plus boxes; plus fields and boxes; everything.
  Fields and boxes alone show what *structure* can do when the printed name is unreliable.
* **Which source?** The ground truth (noise-free evidence) against the unguided Gemini call (the evidence the gate
  actually sees). The gap is the cost of extraction noise.

Metrics per page:

* `top1_acc`: the vendor with the highest $P(\text{vendor} \mid e) = \sum_{L \in \text{vendor}} P(L \mid e)$ is
  the true one
* `mean_p_true`: the average probability given to the true vendor
* `log_loss`: $-\log P(\text{true vendor} \mid e)$, which punishes confident mistakes

In [ ]:
TERM_SETS = {"prior only": ("prior",), "name": ("prior", "name"), "fields": ("prior", "fields"),
             "boxes": ("prior", "boxes"), "fields + boxes": ("prior", "fields", "boxes"), "all": TERMS}


def identification_rows(doc_ids: list[str], evidence: dict[str, dict], source: str,
                        T: float = FIELD_TEMPERATURE) -> list[dict]:
    """Per page and evidence set: the MAP vendor, and the probability given to the true vendor (NEW if unseen)."""
    rows = []
    for doc_id in doc_ids:
        ev = evidence[doc_id]
        truth = records[doc_id]["vendor_key"] if train_count[records[doc_id]["vendor_key"]] else NEW
        for page in ev["pages"]:
            scores = layout_scores(page, ev["vendor_name"], T=T)
            for set_name, terms in TERM_SETS.items():
                vp = vendor_posterior(scores, posterior(scores, terms))
                best = max(vp, key=vp.get)
                rows.append({"doc_id": doc_id, "page": page["number"], "source": source, "evidence": set_name,
                             "true_vendor": truth, "map_vendor": best, "correct": best == truth,
                             "p_map": vp[best], "p_true": vp.get(truth, 0.0)})
    return rows


def identification_summary(df: pd.DataFrame) -> pd.DataFrame:
    return df.groupby(["source", "evidence"], sort=False).agg(
        pages=("correct", "size"), top1_acc=("correct", "mean"), mean_p_true=("p_true", "mean"),
        log_loss=("p_true", lambda p: float(np.mean(-np.log(np.clip(p, 1e-12, 1)))))).round(3)


def reliability(prob: pd.Series, outcome: pd.Series, bins=(0, 0.2, 0.4, 0.6, 0.8, 0.9, 1.0001)) -> pd.DataFrame:
    """Binned predicted probability against the observed frequency: calibrated means the two columns agree."""
    df = pd.DataFrame({"p": prob.astype(float), "y": outcome.astype(float)})
    df["bin"] = pd.cut(df.p, bins, right=False)
    return df.groupby("bin", observed=True).agg(n=("y", "size"), mean_predicted=("p", "mean"),
                                                observed=("y", "mean")).round(3)


part1_docs = [d for d in run_ids if records[d]["vendor_key"] in strong_vendors]
id_df = pd.DataFrame(identification_rows(part1_docs, records, "ground truth") +
                     identification_rows(part1_docs, unguided, "unguided call"))
print(f"Part 1: {len(part1_docs)} test documents from {len(strong_vendors)} strong-tree vendors")
display(identification_summary(id_df))
print("Reliability of P(MAP vendor | e), all evidence, unguided call:")
full = id_df[(id_df.source == "unguided call") & (id_df.evidence == "all")]
reliability(full.p_map, full.correct)

**Extraction under the assumption.** These are the same strong-rule entities, scored for both passes. When the
group is identified correctly, the gated pass should show the same strong rules as the ungated pass, so
`rule_in_range` should match. Any differences in value and box accuracy come from how the rules are presented:
a search range and a confidence instead of a point box.

In [ ]:
strong_rule_keys = {(L["vendor"], L["page"], k) for L in VENDOR_LAYOUTS.values()
                    for k, r in L["rules"].items() if r["strength"] == "strong"}
part1_fields = fields_df[fields_df.doc_id.isin(part1_docs) & fields_df.apply(
    lambda r: (r.vendor_key, min(r.page, 2), r.entity) in strong_rule_keys, axis=1)]
P1_METRICS = ["value_exact_acc", "mean_bbox_overlap", "bbox_overlap>=0.5", "center_within_tol", "mean_rule_vs_gt",
              "rule_in_range", "mean_pred_vs_rule"]
print(f"{part1_fields[part1_fields.setup == 'gated'].shape[0]} strong-rule entity rows")
display(part1_fields.groupby("setup").apply(summarize, include_groups=False)[P1_METRICS].round(3))
setup_table(part1_fields, ["vendor_key"], ["value_exact_acc", "mean_bbox_overlap", "center_within_tol",
                                            "rule_in_range"]).round(3)

### Part 2: all buckets, no assumption
Rules are no longer assumed right. A rule can fail because the page isn't from $L^*$, *or* because the value
moved even though the page is from $L^*$. The full confidence
$\text{conf}_k = \sum_L P(L \mid e)\,P(c_k \in R_{L^*,k} \mid L)$ has to account for both.

**Rule calibration.** For every rule the gated guide considered, withheld ones included, compare its confidence
with whether the ground-truth centre actually fell in its range. Only entities that are printed (filled in ground
truth with a usable box) are used. The confidence is conditional on the value being printed, because it is a
statement about *where*, not *whether*. If the model is calibrated, rules with confidence 0.8 should hit 80% of
the time.

* **Brier score.** $\frac{1}{n}\sum (\text{conf} - \text{hit})^2$: lower is better, and 0.25 is what always
  saying 0.5 earns.
* **ECE.** The count-weighted mean gap between predicted and observed in the reliability table.

In [ ]:
def rule_rows(doc_ids: list[str], T: float = FIELD_TEMPERATURE) -> tuple[pd.DataFrame, pd.DataFrame]:
    """(one row per considered rule with a confidence, one row per page) from the unguided evidence. Recomputed
    here rather than read from the guide files, so the temperature can be swept without new Gemini calls."""
    rules, pages = [], []
    for doc_id in doc_ids:
        ev, gt_pages = unguided[doc_id], {p["number"]: p for p in records[doc_id]["pages"]}
        vendor = records[doc_id]["vendor_key"]
        for page in ev["pages"]:
            g = page_rules(page, ev["vendor_name"], T=T)
            pages.append({"doc_id": doc_id, "page": page["number"], "vendor_key": vendor, "p_new": g["p_new"],
                          "map_is_new": g["chosen"] == NEW,
                          "map_vendor_correct": g["chosen_vendor"] == (vendor if train_count[vendor] else NEW)})
            gt = gt_pages.get(page["number"])
            for r in g["rules"]:
                f = gt["fields"].get(r["name"]) if gt else None
                if r["confidence"] is None or not f or not f["usable"]:
                    continue
                rules.append({"doc_id": doc_id, "page": page["number"], "entity": r["name"], "vendor_key": vendor,
                              "confidence": r["confidence"], "gate": r["gate"], "strength": r["strength"],
                              "hit": H.in_range(H.bbox_center(f["bbox"]), r["range"])})
    rules_df_, pages_df = pd.DataFrame(rules), pd.DataFrame(pages)
    for df in (rules_df_, pages_df):
        df["train_set_size_bucket"] = df.vendor_key.map(lambda v: size_bucket(train_count[v]))
    return rules_df_, pages_df


def calibration_scores(df: pd.DataFrame) -> dict[str, float]:
    table = reliability(df.confidence, df.hit)
    return {"rules": len(df), "brier": float(np.mean((df.confidence - df.hit) ** 2)),
            "ece": float((table.n * (table.mean_predicted - table.observed).abs()).sum() / table.n.sum())}


gate_rules, gate_pages = rule_rows(run_ids)
print({k: round(v, 3) for k, v in calibration_scores(gate_rules).items()})
display(reliability(gate_rules.confidence, gate_rules.hit))
print("Hit rate by gate decision and v2 strength label:")
gate_rules.pivot_table(index="strength", columns="gate", values="hit", aggfunc=["mean", "size"]).round(3)

**Risk–coverage.** Each threshold $\tau$ trades **coverage** (the share of rules shown with a position) against
**accuracy** (the share of shown rules whose range holds the value). An informative confidence makes accuracy rise
as $\tau$ rises. If the curve is flat, the gate isn't telling good rules from bad ones. This table is what
`GATE_BOX` and `GATE_LABEL` should be read against.

**Temperature.** Naive Bayes counts correlated fields as independent evidence, which pushes posteriors toward 0 and
1. If that is happening, a larger $T$ should lower the Brier score and ECE.

In [ ]:
coverage = pd.DataFrame([{"tau": tau, "coverage": (gate_rules.confidence >= tau).mean(),
                          "accuracy_of_shown": gate_rules[gate_rules.confidence >= tau].hit.mean(),
                          "accuracy_of_withheld": gate_rules[gate_rules.confidence < tau].hit.mean()}
                         for tau in np.arange(0.1, 1.0, 0.1).round(1)])
display(coverage.round(3))

sweep = []
for T in (1.0, 2.0, 4.0):
    r_, p_ = rule_rows(run_ids, T=T)
    sweep.append({"T": T, **calibration_scores(r_), "share_box": (r_.gate == "box").mean(),
                  "box_accuracy": r_[r_.gate == "box"].hit.mean(), "map_vendor_acc": p_.map_vendor_correct.mean()})
pd.DataFrame(sweep).round(3)

**Per training-set size bucket.** These are extraction metrics for both passes, plus what the gate did in each
bucket:

* `mean_p_new`: the average P(NEW | e)
* `share_pages_new`: the share of pages whose MAP was NEW, so that only corpus labels were shown
* `map_vendor_acc`: the share of pages whose MAP layout belongs to the true vendor (or is NEW for an unseen vendor)
* `share_box` / `share_label` / `share_withheld`: the gate decisions over the rules considered

The `0 (unseen)` row is the main test. There v2's guide handed over a stranger's layout and lost 0.13 of
`bbox_overlap`. A working gate should recognise those pages as NEW and leave the memory out.

In [ ]:
BUCKET_METRICS = ["value_exact_acc", "false_fill", "mean_bbox_overlap", "center_within_tol", "rule_in_range",
                  "mean_pred_vs_rule"]
gate_stats = gate_rules.groupby("train_set_size_bucket").agg(
    mean_conf=("confidence", "mean"), share_box=("gate", lambda g: (g == "box").mean()),
    share_label=("gate", lambda g: (g == "label").mean()), share_withheld=("gate", lambda g: (g == "withheld").mean()))
page_stats = gate_pages.groupby("train_set_size_bucket").agg(
    pages=("page", "size"), mean_p_new=("p_new", "mean"), share_pages_new=("map_is_new", "mean"),
    map_vendor_acc=("map_vendor_correct", "mean"))
by_bucket = (page_stats.join(gate_stats).join(setup_table(fields_df, ["train_set_size_bucket"], BUCKET_METRICS))
             .reindex([b for b in BUCKET_ORDER if b in page_stats.index]))
by_bucket.round(3)

In [ ]:
per_vendor = (vendors_df.set_index("vendor_key")[["n_train", "n_test", "train_set_size_bucket"]]
              .join(gate_pages.groupby("vendor_key").agg(mean_p_new=("p_new", "mean"),
                                                         map_vendor_acc=("map_vendor_correct", "mean")))
              .join(gate_rules.groupby("vendor_key").agg(mean_conf=("confidence", "mean"),
                                                         share_box=("gate", lambda g: (g == "box").mean())))
              .join(setup_table(fields_df, ["vendor_key"], ["value_exact_acc", "mean_bbox_overlap",
                                                            "center_within_tol"]), how="inner")
              .sort_values("n_train", ascending=False))
per_vendor.round(3)

### Pass 1 vs pass 2: which documents changed most?
`compare_passes` diffs the two saved output files of every document entity by entity. A **change** is a different
value, or a box that moved (`bbox_overlap` between the pass-1 and pass-2 boxes below 0.5). Each change is judged
against ground truth:

* **improved**: pass 2 is right where pass 1 was wrong, or the box moved closer to the ground truth
* **regressed**: the reverse
* **neutral**: anything else

Changes are also split by what the gate did with that entity's rule. Regressions under `withheld` mean pass 1's
rule was helping and the gate removed it. Improvements under `withheld` mean the gate removed a misleading rule.
`difference_score = n_fields_changed + Σ(1 − bbox_overlap(pass1, pass2))` ranks the documents.

In [ ]:
def compare_passes(pass1_dir: Path, pass2_dir: Path, doc_ids: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    """(per-document summary, per-field changes) between two directories of saved pass outputs."""
    changes, summary = [], []
    for doc_id in doc_ids:
        facts = page_facts[doc_id]
        p1, p2 = load_pass(pass1_dir, doc_id, facts), load_pass(pass2_dir, doc_id, facts)
        if p1 is None or p2 is None:
            continue
        gt = records[doc_id]
        p2_pages = {p["number"]: p for p in p2["pages"]}
        gt_pages = {p["number"]: p for p in gt["pages"]}
        shifts, doc_changes = [], []
        for page1 in p1["pages"]:
            n = page1["number"]
            for key in BOX_KEYS:
                a = page1["fields"].get(key, {"value": "", "bbox": None})
                b = p2_pages.get(n, {"fields": {}})["fields"].get(key, {"value": "", "bbox": None})
                g = gt_pages.get(n, {"fields": {}})["fields"].get(key, {"value": "", "bbox": None})
                same_value = H.normalize_for_match(a["value"]) == H.normalize_for_match(b["value"])
                box_overlap = H.bbox_overlap(a["bbox"], b["bbox"]) if a["bbox"] and b["bbox"] else \
                    (1.0 if not a["bbox"] and not b["bbox"] else 0.0)
                shifts.append(1 - box_overlap)
                if same_value and box_overlap >= 0.5:
                    continue
                a_ok = H.normalize_for_match(a["value"]) == H.normalize_for_match(g["value"])
                b_ok = H.normalize_for_match(b["value"]) == H.normalize_for_match(g["value"])
                box_gain = H.bbox_overlap(b["bbox"], g["bbox"]) - H.bbox_overlap(a["bbox"], g["bbox"])
                verdict = ("improved" if (b_ok and not a_ok) or (a_ok == b_ok and box_gain > 0.1) else
                           "regressed" if (a_ok and not b_ok) or (a_ok == b_ok and box_gain < -0.1) else "neutral")
                doc_changes.append({"doc_id": doc_id, "page": n, "entity": key, "verdict": verdict,
                                    "pass1_value": a["value"], "pass2_value": b["value"], "gt_value": g["value"],
                                    "pass1_box": H.bbox_to_box2d(a["bbox"]) if a["bbox"] else None,
                                    "pass2_box": H.bbox_to_box2d(b["bbox"]) if b["bbox"] else None,
                                    "gt_box": H.bbox_to_box2d(g["bbox"]) if g.get("bbox") else None})
        changes += doc_changes
        summary.append({"doc_id": doc_id, "vendor_key": gt["vendor_key"], "n_fields_changed": len(doc_changes),
                        "n_improved": sum(c["verdict"] == "improved" for c in doc_changes),
                        "n_regressed": sum(c["verdict"] == "regressed" for c in doc_changes),
                        "difference_score": len(doc_changes) + float(np.sum(shifts))})
    summary_df = pd.DataFrame(summary)
    if not summary_df.empty:
        summary_df = summary_df.sort_values("difference_score", ascending=False).reset_index(drop=True)
    return summary_df, pd.DataFrame(changes)


diff_summary, diff_changes = compare_passes(RESULTS_DIR / "pass1", RESULTS_DIR / "pass2", run_ids)
display(diff_summary.head(15).round(3))
if not diff_changes.empty:
    gate_lookup = gate_rules.set_index(["doc_id", "page", "entity"]).gate
    diff_changes["gate"] = [gate_lookup.get((r.doc_id, r.page, r.entity), "no rule considered")
                            for r in diff_changes.itertuples()]
    display(pd.crosstab(diff_changes.gate, diff_changes.verdict, margins=True))
    for row in diff_summary.head(TOP_DIFF_DOCS).itertuples():
        print(f"\n=== {row.doc_id}  vendor={row.vendor_key}  changed={row.n_fields_changed}  "
              f"improved={row.n_improved}  regressed={row.n_regressed}  score={row.difference_score:.2f}")
        display(diff_changes[diff_changes.doc_id == row.doc_id].drop(columns="doc_id"))

**Reading the results.**
* **Part 1, low top-1 accuracy with fields alone but high with fields and boxes.** Many vendors print the same
  fields, so only positions tell their trees apart. Low accuracy with *all* evidence on the unguided call, against
  high accuracy on ground truth, means extraction noise is the bottleneck: widen `BOX_RANGE_MIN` or raise
  `FIELD_TEMPERATURE`.
* **Part 2, reliability table above the diagonal** (observed > predicted): the gate is too cautious, so lower
  `GATE_BOX`. **Below the diagonal**: overconfident, so raise $T$ or `FIELD_SMOOTHING_M`.
* **`0 (unseen)` bucket.** High `share_pages_new` with a `mean_bbox_overlap_gain` near or above 0 means the gate
  stopped v2's loss. A low `share_pages_new` means `NAME_ERROR` or the prior let a stranger's layout win.
* **Regressions in the `withheld` row** of the change table are the price of the gate: rules that would have
  helped but weren't trusted.

In [ ]:
driver.close()